# Mögliche Features

**Primäre Korrelate:**

 - Masseter-Querschnitt        → größter Kaumuskel, direkter Korrelationist
  
 - Kortikalisdicke Mandibula   → Knochen überträgt Kaukraft
  
 - Gonialwinkel                → flacher Winkel = mehr Hebelkraft
  
 - Ramus-Höhe und -Breite      → biomechanischer Hebel
  
 - Kondylenmorphologie         → Kraftübertragung ins Gelenk

**Sekundäre Korrelate:**

 - Zahnanzahl / Okklusion      → Kraftverteilung
  
 - Mandibula-Körper-Breite     → Querschnittsfläche = Kraftkapazität
  
 - Trabekelstruktur            → Knochenqualität, Mineraldichte

# Mandibula-fokussierter ROI-Crop

In [ ]:
def mandible_focused_crop(image_tensor: torch.Tensor,
                          roi_size: tuple = (96, 96, 96)) -> torch.Tensor:
    """
    Crop fokussiert auf Mandibula + Masseter-Region.

    Anatomische Begründung:
      Ganzkopf-Crop: ~70% Luft + Calvarium — nicht relevant für Kaukraft
      Mandibula-Crop: enthält Kortikalis, Masseter-Ansatz, Zähne
      → Encoder sieht ausschließlich kaukraftrelevante Strukturen

    Strategie:
      Unteres 60% des Schädels enthält Mandibula + Masseter.
      Crop aus dieser Region garantiert relevante Anatomie.
    """
    C, D, H, W = image_tensor.shape

    # Mandibula liegt im unteren 60% der Schädelhöhe (Dimension H)
    mandible_start_h = int(H * 0.40)  # ab 40% der Höhe
    roi_d, roi_h, roi_w = roi_size

    # Zufälligen Crop innerhalb der Mandibula-Region
    d_start = torch.randint(0, max(1, D - roi_d), (1,)).item()
    h_start = torch.randint(mandible_start_h,
                             max(mandible_start_h + 1, H - roi_h), (1,)).item()
    w_start = torch.randint(0, max(1, W - roi_w), (1,)).item()

    return image_tensor[:, d_start:d_start+roi_d,
                           h_start:h_start+roi_h,
                           w_start:w_start+roi_w]

# Knochen-Fenster Preprocessing (HU-Windowing)

In [ ]:
def bone_window_transform(image_tensor: torch.Tensor,
                           window_center: float = 0.55,
                           window_width: float = 0.5) -> torch.Tensor:
    """
    Simuliert CT-Knochenfenster nach Normalisierung.

    Klinischer Hintergrund:
      Radiologen verwenden Knochenfenster (W=1500 HU, L=300 HU)
      um Kortikalis-Dicke und Trabekel-Struktur zu beurteilen.
      Beide Strukturen korrelieren direkt mit Kaukraft.

    Nach Percentile-Normalisierung auf [0,1]:
      Luft:       ~0.0
      Weichgewebe: ~0.3-0.5
      Kortikalis:  ~0.6-0.8  ← wichtig für Kaukraft
      Schmelz:     ~0.9-1.0

    window_center=0.55, window_width=0.5:
      → fokussiert auf Knochen/Gewebe-Grenzbereich
      → Kontrasterhöhung genau im kaukraftrelevanten Bereich
    """
    lo = window_center - window_width / 2
    hi = window_center + window_width / 2
    windowed = (image_tensor - lo) / (window_width + 1e-8)
    return windowed.clamp(0.0, 1.0)


# In aug_mae / aug_local einbauen:
from monai.transforms import ScaleIntensityRanged

bone_window = ScaleIntensityRanged(
    keys=["image"],
    a_min=0.30, a_max=0.80,   # Weichgewebe bis Kortikalis
    b_min=0.0,  b_max=1.0,
    clip=True,
)

# Nur anatomisch valide Flips — ausschließlich links/rechts

In [ ]:
# FALSCH für Kaukraft: alle drei Achsen flippen
# Front-Back Flip: Zähne zeigen nach hinten — anatomisch unmöglich
# Top-Bottom Flip: Mandibula oben — anatomisch unmöglich

# RICHTIG: nur Links-Rechts Flip
# Begründung: Kiefer ist annähernd bilateral symmetrisch
#             Kaukraft ist links/rechts ähnlich
#             Flip verdoppelt effektiv den Datensatz

# In aug_local:
RandFlipd(
    keys=["image"],
    prob=0.5,
    spatial_axis=2,   # NUR Links-Rechts (W-Achse) — NICHT 0 oder 1
)

# spatial_axis=0: Sagittal-Flip → Mandibula hinten = FALSCH
# spatial_axis=1: Vertikal-Flip → Mandibula oben  = FALSCH
# spatial_axis=2: Links-Rechts  → anatomisch valide ✅

# Knochen-Dichte Perturbation — simuliert Mineralisierungsunterschiede

In [ ]:
def bone_density_perturbation(image_tensor: torch.Tensor,
                               bone_threshold: float = 0.5,
                               delta_range: float = 0.15) -> torch.Tensor:
    """
    Verschiebt die Knochen-Intensität selektiv.

    Biomechanische Begründung:
      Knochenmineral-Dichte (BMD) korreliert mit Kaukraft.
      Patienten mit höherer BMD haben stärkere Kaumuskeln.
      Diese Augmentierung simuliert BMD-Variation und zwingt
      den Encoder relative Muster statt absolute HU-Werte zu lernen.

    Wichtig: NUR Knochen-Voxel werden verschoben, nicht Luft/Weichgewebe.
    Dadurch bleibt die Knochen/Weichgewebe-Grenze erhalten.
    """
    delta = (torch.rand(1).item() - 0.5) * 2 * delta_range
    bone_mask = (image_tensor > bone_threshold).float()
    return (image_tensor + delta * bone_mask).clamp(0.0, 1.0)

# Als MONAI Lambda Transform:
from monai.transforms import Lambda

BoneDensityAug = Lambda(
    func=lambda x: bone_density_perturbation(x, bone_threshold=0.5, delta_range=0.15)
)

# Elastische Deformation — simuliert Kiefermorphologie-Varianz

In [ ]:
# Kleine elastische Deformation simuliert natürliche
# interindividuelle Variation der Kiefermorphologie

RandAffined(
    keys=["image"],
    prob=0.6,
    # KLEIN halten — Kiefermorphologie nicht zerstören
    rotate_range=(0.05, 0.05, 0.08),   # max 4.5 Grad — war 0.15 (8.5 Grad)
    translate_range=(2, 2, 2),          # 2 Voxel max
    scale_range=(0.03, 0.03, 0.03),    # ±3% Skalierung
    mode="bilinear",
    padding_mode="zeros",
)

# Begründung der Reduktion:
# Zu starke Rotation/Translation → Gonialwinkel und
# Ramus-Proportionen werden zerstört
# Diese geometrischen Maße korrelieren direkt mit Kaukraft

# Lokale Kortikalis-Textur-Augmentierung

In [ ]:
from monai.transforms import RandGaussianSmoothd

# Leichtes zufälliges Smoothing simuliert verschiedene CT-Schärfen
# zwischen HiRes-Gerät und Boen-Gerät

RandGaussianSmoothd(
    keys=["image"],
    sigma_x=(0.3, 1.5),
    sigma_y=(0.3, 1.5),
    sigma_z=(0.3, 1.5),
    prob=0.4,
)

# Begründung:
# HiRes 3D-Plus: höhere Auflösung → schärfere Kortikalis
# Boen Zhongding: andere Schärfe
# Smoothing-Augmentierung macht Encoder robust gegenüber
# gerätebedingten Texturunter-schieden — der Encoder lernt
# strukturelle Merkmale statt Geräte-Artefakte

# Altersgruppen-stratifizierte Normalisierung

In [ ]:
def age_stratified_normalize(image_tensor: torch.Tensor,
                              age: int) -> torch.Tensor:
    """
    Verschiedene Normalisierungsparameter für Altersgruppen.

    Klinischer Hintergrund:
      Kinderschädel (5-17):    weniger Mineralisation, dünnere Kortikalis
      Erwachsene (18-60):      maximale Knochendichte, stabile Morphologie
      Ältere (60+):            Knochenabbau, veränderte Trabekelung

    Ohne altersgruppen-spezifische Normalisierung:
      Ein einheitliches Percentile-Clipping behandelt alle gleich
      → Encoder kann Alter-Bias nicht von Kaukraft-Signal trennen

    Mit stratifizierter Normalisierung:
      Altersgruppen-spezifische Intensitätsbereiche werden harmonisiert
      → Encoder muss strukturelle Merkmale lernen, nicht Helligkeits-Proxies
    """
    if age < 18:
        # Jugendliche: Knochen heller normalisieren (weniger Mineralisierung)
        lo, hi = 0.02, 0.95
    elif age < 60:
        # Erwachsene: Standard
        lo, hi = 0.01, 0.99
    else:
        # Ältere: Knochen dunkler (mehr Porosität)
        lo, hi = 0.01, 0.97

    p_lo = torch.quantile(image_tensor[image_tensor > 0.01], lo)
    p_hi = torch.quantile(image_tensor[image_tensor > 0.01], hi)
    return ((image_tensor - p_lo) / (p_hi - p_lo + 1e-8)).clamp(0.0, 1.0)

# Bilateral-ähnliches Edge-Preserving Smoothing

In [ ]:
def edge_preserving_smooth(image_tensor: torch.Tensor,
                            iterations: int = 2) -> torch.Tensor:
    """
    Rauschreduktion OHNE Kortikalis-Kanten zu zerstören.

    Warum wichtig:
      Standard Gaussian Smoothing verschmiert Kortikalis-Kanten.
      Kortikalis-Dicke ist ein primärer Kaukraft-Prädiktor.
      Edge-Preserving Smoothing reduziert Rauschen in homogenen
      Bereichen (Spongiosa, Luft) ohne Kanten zu verwischen.

    Implementierung: Median-Filter als Approximation (schneller als bilateral)
    """
    import torch.nn.functional as F

    kernel_size = 3
    padding = kernel_size // 2

    for _ in range(iterations):
        # Unsharp masking Ansatz: Original - Smooth + Original
        smooth = F.avg_pool3d(
            image_tensor.unsqueeze(0),
            kernel_size=kernel_size,
            stride=1,
            padding=padding,
        ).squeeze(0)

        # Edge-Stärke bestimmen
        edge_strength = (image_tensor - smooth).abs()
        edge_mask     = (edge_strength > 0.05).float()

        # Original an Kanten, smooth in homogenen Bereichen
        image_tensor = image_tensor * edge_mask + smooth * (1 - edge_mask)

    return image_tensor.clamp(0.0, 1.0)

# Multi-Scale Dual-Crop — grobe Morphologie + feine Textur

In [ ]:
def create_multiscale_mae_input(image_tensor: torch.Tensor,
                                 coarse_size: tuple = (64, 64, 64),
                                 fine_size:   tuple = (32, 32, 32)
                                 ) -> torch.Tensor:
    """
    Kombiniert zwei Crop-Scales zu einem größeren Input.

    Biomechanische Begründung:
      Kaukraft = f(globale Morphologie, lokale Knochenqualität)

      Grober Crop (64³):
        → Gonialwinkel, Ramus-Geometrie, Gesamtform
        → 'Welche Form hat der Kiefer?'

      Feiner Crop (32³) aus Mandibula-Region:
        → Kortikalis-Dicke, Trabekel-Muster, Dichte
        → 'Wie stark ist der Knochen?'

      Verkettung beider Crops:
        → Encoder lernt gleichzeitig Form UND Qualität
        → beides ist für Kaukraft-Vorhersage nötig
    """
    # Grober Crop: zufällig im unteren Schädeldrittel
    coarse = mandible_focused_crop(image_tensor, coarse_size)

    # Feiner Crop: aus demselben Bereich, höhere 'Zoom'-Stufe
    fine = mandible_focused_crop(image_tensor, fine_size)

    # Fine auf coarse_size hochskalieren
    fine_up = F.interpolate(
        fine.unsqueeze(0), size=coarse_size,
        mode="trilinear", align_corners=False
    ).squeeze(0)

    # Kanäle verketten → [2, D, H, W]
    return torch.cat([coarse, fine_up], dim=0)

# Muskel-Region Dropout — erzwingt Knochen-Feature-Lernen

In [ ]:
def muscle_region_dropout(image_tensor: torch.Tensor,
                           prob: float = 0.3,
                           soft_tissue_threshold: float = 0.15,
                           hard_tissue_threshold: float = 0.45) -> torch.Tensor:
    """
    Löscht Weichgewebe-Regionen selektiv aus.

    Trainings-Strategie:
      Problem: Masseter-Muskelgröße korreliert mit Kaukraft
               → Encoder könnte 'Weichgewebe-Größe schätzen' lernen
               → Das ist eine Abkürzung, keine strukturelle Eigenschaft

      Lösung:  Zufälliges Ausblenden der Weichgewebe-Region
               → Encoder muss Knochen-Morphologie lernen
               → Kortikalis-Geometrie statt Muskelvolumen

    Achtung: prob=0.3 — nicht zu oft anwenden,
             Muskelgröße ist ein valides Signal für Kaukraft
    """
    if torch.rand(1).item() > prob:
        return image_tensor

    # Weichgewebe-Maske: zwischen Luft und Knochen
    soft_mask = (
        (image_tensor > soft_tissue_threshold) &
        (image_tensor < hard_tissue_threshold)
    ).float()

    # Zufälliges teilweises Ausblenden (nicht 100% — zu aggressiv)
    dropout_mask = (torch.rand_like(image_tensor) > 0.7).float()
    combined     = soft_mask * dropout_mask

    return image_tensor * (1.0 - combined)